In [ ]:
name = "granite"

In [ ]:
from docling.datamodel import vlm_model_specs
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.pipeline.vlm_pipeline import VlmPipeline

source = "input.pdf"
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_cls=VlmPipeline,
        ),
    }
)
doc = converter.convert(source=source).document
print(doc.export_to_markdown())
# Or export as HTML, JSON, etc.


In [9]:
from pdf2image import convert_from_path
from PIL import Image
import os
def extract_first_page_as_image(pdf_path, dpi=150):
    """Extract first page from PDF using pdf2image"""
    # Convert only the first page (pages parameter is 1-indexed)
    pages = convert_from_path(pdf_path, dpi=dpi, first_page=1, last_page=1)
    return pages[0]  # Returns PIL Image

# Installation
# pip install pdf2image

# Usage
pdf_path = r"C:\Users\User\Projects\scaled_processing\data\documents\raw\GSPP_5407_202507_Billing.pdf"
first_page_image = extract_first_page_as_image(pdf_path)
first_page_image.save("images/first_page.png")


In [ ]:
from transformers import AutoProcessor, AutoModelForVision2Seq
processor = AutoProcessor.from_pretrained("ibm-granite/granite-docling-258M")
model = AutoModelForVision2Seq.from_pretrained("ibm-granite/granite-docling-258M")

In [ ]:
import torch
from docling_core.types.doc import DoclingDocument
from docling_core.types.doc.document import DocTagsDocument
from transformers import AutoProcessor, AutoModelForVision2Seq
from transformers.image_utils import load_image
from pathlib import Path

# Device setup
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load your page image (can be from PDF first page)
image = load_image(r"C:\Users\User\Projects\scaled_processing\data\images\first_page.png")  # or load_image(url)

# Initialize processor and model
processor = AutoProcessor.from_pretrained("ibm-granite/granite-docling-258M")
model = AutoModelForVision2Seq.from_pretrained(
    "ibm-granite/granite-docling-258M",
    torch_dtype=torch.bfloat16,
    _attn_implementation="sdpa" if DEVICE == "cuda" else "sdpa",
).to(DEVICE)

# Create input messages
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Convert this page to docling."}
        ]
    }
]

# Prepare inputs
prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[image], return_tensors="pt")
inputs = inputs.to(DEVICE)

# Generate outputs
generated_ids = model.generate(**inputs, max_new_tokens=8192)
prompt_length = inputs.input_ids.shape[1]
trimmed_generated_ids = generated_ids[:, prompt_length:]

# Get DocTags (structured intermediate format)
doctags = processor.batch_decode(
    trimmed_generated_ids,
    skip_special_tokens=False,
)[0].lstrip()

print(f"DocTags: \n{doctags}\n")

# Convert to structured document
doctags_doc = DocTagsDocument.from_doctags_and_image_pairs([doctags], [image])
doc = DoclingDocument.load_from_doctags(doctags_doc, document_name="Document")

# Export structured outputs
print(f"Markdown:\n{doc.export_to_markdown()}\n")

# Save as various formats
Path("out/").mkdir(parents=True, exist_ok=True)
doc.save_as_html(Path("out/example.html"))
doc.save_as_markdown(Path("out/example.md"))
# JSON output available via: doc.export_to_json()
